# Algoritmos de Regresión

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Semana12_Supervisado_G5").getOrCreate()
df_clusters = spark.read.parquet("/home/jovyan/work/datos_etiquetados_alojamientos")
df_supervisado = df_clusters.withColumnRenamed("prediction", "label")
print(f"Total: {df_supervisado.count()}")
df_supervisado.groupBy("label").count().orderBy("label").show()

Total: 4408
+-----+-----+
|label|count|
+-----+-----+
|    0| 1253|
|    1|  494|
|    2| 1120|
|    3|  917|
|    4|  624|
+-----+-----+



In [2]:
train_data, test_data = df_supervisado.randomSplit([0.7, 0.3], seed=42)
from pyspark.ml.feature import VectorAssembler, StandardScaler
feature_cols = ["precio_num","puntuacion_num","estrellas_num","ciudad_cat","zona_cat","tipo_cat","plataforma_cat"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_vec = assembler.transform(train_data)
test_vec = assembler.transform(test_data)
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
scaler_model = scaler.fit(train_vec)
train_scaled = scaler_model.transform(train_vec)
test_scaled = scaler_model.transform(test_vec)

In [7]:
# Configurar el modelo de Regresión Lineal
lr_regresion = LinearRegression(
    featuresCol="scaledFeatures_regresion", 
    labelCol="label_precio", 
    maxIter=10
)

# Entrenar el modelo con los datos de entrenamiento
lr_reg_model = lr_regresion.fit(train_reg)

# Hacer las predicciones sobre los datos de prueba
predictions_regresion = lr_reg_model.transform(test_reg)

# Mostrar las predicciones junto al precio real
print("=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===")
predictions_regresion.select("marca", "label_precio", "prediction").show(10)

=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===
+-----+------------------+-----------------+
|marca|      label_precio|       prediction|
+-----+------------------+-----------------+
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
+-----+------------------+-----------------+
only showing top 10 rows



In [3]:
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
dt = DecisionTreeClassifier(featuresCol="scaledFeatures", labelCol="label", maxDepth=5, seed=42)
dt_model = dt.fit(train_scaled)
dt_pred = dt_model.transform(test_scaled)
evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
acc_dt = evaluator.evaluate(dt_pred)
print(f"Árbol Decisión: {acc_dt*100:.2f}%")

Árbol Decisión: 97.16%


In [4]:
from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(featuresCol="scaledFeatures", labelCol="label", numTrees=20, seed=42)
rf_model = rf.fit(train_scaled)
rf_pred = rf_model.transform(test_scaled)
acc_rf = evaluator.evaluate(rf_pred)
print(f"Random Forest: {acc_rf*100:.2f}%")


Random Forest: 96.78%
